## LIBRARIES

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

## WIDGETS

In [0]:
dbutils.widgets.removeAll()

In [0]:

dbutils.widgets.text("storageName", "saccexplorer")
dbutils.widgets.text("containerName", "silver")

dbutils.widgets.text("catalogName", "unit_catalog_explorer")
dbutils.widgets.text("schema_source", "uc_bronze")
dbutils.widgets.text("schema_sink", "uc_silver")


## CONSTANTS

In [0]:
storage = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("containerName")

catalog         = dbutils.widgets.get("catalogName")
schema_source   = dbutils.widgets.get("schema_source")
schema_sink     = dbutils.widgets.get("schema_sink")

table_ranking   = "supercias_ranking"
table_compania  = "supercias_compania"
table_segmento  = "supercias_segmento"
table_sector    = "supercias_sector"

table_ranking_scored    = "ranking_enriquecido"
table_resumen_provincia = "ranking_por_provincia"
table_resumen_sector    = "ranking_por_sector"


## PATHS

In [0]:
path_base_silver = f"abfss://{container}@{storage}.dfs.core.windows.net/{schema_sink}"

path_final_scored = f"{path_base_silver}/{table_ranking_scored}"
path_final_prov   = f"{path_base_silver}/{table_resumen_provincia}"
path_final_sect   = f"{path_base_silver}/{table_resumen_sector}"

## READ AND CACHE

In [0]:
df_compania = spark.table(f"{catalog}.{schema_source}.{table_compania}")
df_segmento = spark.table(f"{catalog}.{schema_source}.{table_segmento}")
df_sector   = spark.table(f"{catalog}.{schema_source}.{table_sector}")


# Caching para optimizar los joins
df_compania.cache()
df_segmento.cache()
df_sector.cache()

## UDF

In [0]:
# --- UDF: Calificación de Desempeño ---
# Basado en la utilidad neta para categorizar la rentabilidad
def score_desempeño_empresa(utilidad):
    if utilidad is None: return "SIN DATOS"
    if utilidad >= 10000000:  # 10 Millones
        return "TOP PERFORMANCE"
    elif utilidad >= 1000000: # 1 Millon
        return "SÓLIDA"
    elif utilidad > 0:
        return "ESTABLE"
    else:
        return "EN RIESGO"

score_desempeño_udf = F.udf(score_desempeño_empresa, StringType())

## JOINS

In [0]:
df_ranking = spark.table(f"{catalog}.{schema_source}.{table_ranking}")

# Limpieza de llaves (Trims)
df_ranking_clean = df_ranking.withColumn("ciiu_n1", trim(col("ciiu_n1")))
df_sector_clean  = df_sector.withColumn("ciiu", trim(col("ciiu")))

# Join Maestro (Ranking + Dimensiones)
df_enriquecido = df_ranking_clean.alias("r") \
    .join(broadcast(df_compania.alias("c")), col("r.expediente") == col("c.expediente")) \
    .join(broadcast(df_segmento.alias("s")), col("r.cod_segmento") == col("s.id_segmento"), "left") \
    .join(broadcast(df_sector_clean.alias("sec1")), col("r.ciiu_n1") == col("sec1.ciiu"), "left") \
    .join(broadcast(df_sector_clean.alias("sec2")), col("r.ciiu_n6") == col("sec2.ciiu"), "left") \
    .select(
        "r.*", 
        "c.ruc", "c.nombre", "c.tipo","c.provincia", 
        "s.*",
        col("sec1.ciiu").alias("ciiu_general"),
        col("sec1.descripcion").alias("sector_economico_general"),
        col("sec2.ciiu").alias("ciiu_especifico"),
        col("sec2.descripcion").alias("sector_economico_especifico")
    )

## KPI'S

In [0]:
# Aplicar UDF de Score
df_enriquecido = df_enriquecido.withColumn("calificacion_rentabilidad", score_desempeño_udf(col("utilidad_neta")))


# Resumen por Provincia y Año
df_provincia_kpi = (df_enriquecido.groupBy("anio", "provincia")
                    .agg(F.sum("ingresos_ventas").alias("total_ventas"),
                         F.avg("utilidad_neta").alias("promedio_utilidad"),
                         F.count("expediente").alias("num_companias"))
                    .orderBy("anio", desc("total_ventas")))

# Resumen por Sector Económico
df_sector_kpi = (df_enriquecido.groupBy("anio", "sector_economico_general")
                    .agg(F.sum("activos").alias("total_activos"),
                         F.sum("patrimonio").alias("total_patrimonio"))
                    .orderBy("anio", desc("total_activos")))

## SAVE

In [0]:

# --- 2. Guardar Tabla Maestra Enriquecida ---
df_enriquecido.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("path", path_final_scored) \
    .saveAsTable(f"{catalog}.{schema_sink}.{table_ranking_scored}")

# --- 3. Guardar KPI Provincia ---
df_provincia_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_final_prov) \
    .saveAsTable(f"{catalog}.{schema_sink}.{table_resumen_provincia}")

# --- 4. Guardar KPI Sector ---
df_sector_kpi.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_final_sect) \
    .saveAsTable(f"{catalog}.{schema_sink}.{table_resumen_sector}")

print("Transformaciones guardadas exitosamente en formato Delta en el Data Lake.")